# Explorarea datelor

Scop: structura, acoperirea și inconsistențele din `companies.jsonl`,
înainte de a scrie orice cod de parsare.

In [5]:
import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

DATA = ROOT / "data" / "raw" / "companies.jsonl"

rows = [json.loads(line) for line in open(DATA, encoding="utf-8") if line.strip()]

failures = Counter()

import sys

sys.path.insert(0, str(ROOT / "src"))
from company_qualifier.ingest.parse import load_companies

companies, failures = load_companies(ROOT / "data" / "raw" / "companies.jsonl")
print(len(companies), "|", dict(failures) or "0 eșecuri")
print(companies[0].operational_name, companies[0].address.country_code, companies[0].primary_naics.code)
from company_qualifier.ingest.parse import load_companies
from company_qualifier.ingest.profile import build_profile
from company_qualifier.ingest.dedupe import dedupe_exact, find_name_groups
companies, failures = load_companies(ROOT / "data" / "raw" / "companies.jsonl")
print("parsate:", len(companies), "| eșecuri:", dict(failures) or 0)

deduped, removed = dedupe_exact(companies)
print("după dedupe:", len(deduped), "| eliminate:", sum(removed.values()))
print("grupuri rămase:", list(find_name_groups(deduped).keys()))

print("\n--- profil normal ---")
print(build_profile(deduped[0]))

print("\n--- profil fără nume ---")
fn = next(c for c in deduped if not c.operational_name)
print(build_profile(fn))

lungimi = sorted(len(build_profile(c)) for c in deduped)
print("\nprofil, caractere: min", lungimi[0], "| mediană", lungimi[len(lungimi)//2], "| max", lungimi[-1])



477 | 0 eșecuri
Rompetrol ro 324110
parsate: 477 | eșecuri: 0
după dedupe: 457 | eliminate: 20
grupuri rămase: ['Rompetrol', 'Versar', 'Sesame HR', 'Decathlon', 'RJETech']

--- profil normal ---
Rompetrol. Petroleum Refineries. Rompetrol is a Romanian company specialized in petroleum refining, petrochemical operations, and the distribution of fuel products. The company operates as a subsidiary of KMG International and manages integrated refineries in Romania, Moldova, Bulgaria, and Georgia, serving the automotive, industrial, and energy sectors. Rompetrol also provides industrial products, wholesale fuel supply, and e-Mobility services, and is certified for quality, health, safety, and environmental management. offerings: Fuel Product Distribution, Quality and Safety Compliance Services, Petrochemical Product Manufacturing, E-Mobility Solutions Implementation, Petroleum Refining and Distribution, Premium Fuel Retail, LPG Gas Production and Distribution, Sustainable Energy Solutions Dev

In [ ]:
for c in companies:
    if not c.operational_name:
        print(c.row_index, "|", c.website, "|", c.address.country_code,
              "|", c.primary_naics.label)
        print("   ", c.description[:200])
        print("   ", c.core_offerings[:3])
        print()


138 | icontributetoml.herokuapp.com | ca | Research and Development in the Physical, Engineering, and Life Sciences (except Nanotechnology and Biotechnology)
    The entity associated with the domain icontributetoml.herokuapp.com is a Canadian research and development organization engaged in the development and dissemination of machine learning algorithms and 
    ['ML Algorithm Taxonomy and Selection Criteria Development', 'Data Science Knowledge Dissemination Services', 'Machine Learning Algorithm Dissemination Services']

263 | seycosmetics.co.uk | us | Toilet Preparation Manufacturing
    Bodis, LLC is a United States company specialized in the production and distribution of beauty cosmetics and cosmetic packaging. The company primarily serves wholesale clients in the United States, of
    ['Beauty Cosmetics Wholesale', 'Beauty Cosmetics Manufacturing', 'Cosmetic Packaging Wholesale']

/home/andre/proiecte/company-qualifier/src/company_qualifier/ingest/dedupe.py
[]


In [ ]:
def is_empty(value) -> bool:
    """Gol = None, string doar cu spații, sau listă/dict fără elemente."""
    if value is None:
        return True
    if isinstance(value, str) and not value.strip():
        return True
    return bool(isinstance(value, (list, dict)) and len(value) == 0)

In [ ]:
FIELDS = sorted({k for r in rows for k in r})
n = len(rows)

for field in FIELDS:
    present = sum(not is_empty(r.get(field)) for r in rows)
    print(f"{field:18s} {present:4d}/{n}  {100*present/n:5.1f}%")

In [ ]:
for field in FIELDS:
    print(f"{field:18s} {dict(Counter(type(r.get(field)).__name__ for r in rows))}")

In [ ]:
print("fără country_code:", sum(1 for r in rows if not r["address"].get("country_code")))
print("fără naics code  :", sum(1 for r in rows if not r["primary_naics"].get("code")))

In [ ]:
tari = Counter(r["address"].get("country_code") for r in rows)
print("țări distincte:", len(tari))
for cod, cate in tari.most_common(20):
    print(f"  {cod}  {cate}")

In [ ]:
sect = Counter(r["primary_naics"]["code"][:2] for r in rows)
print(sect.most_common())
print("\n48 transport:", sect.get("48", 0), "| 49 depozitare:", sect.get("49", 0))

In [ ]:
lungimi = sorted(len(r["description"]) for r in rows)
print("min:", lungimi[0], "| mediană:", lungimi[n//2], "| p95:", lungimi[int(n*0.95)], "| max:", lungimi[-1])

In [ ]:
site = Counter(r["website"] for r in rows if r["website"])
nume = Counter(r["operational_name"] for r in rows if r["operational_name"])
print("website duplicat:", [(k, v) for k, v in site.most_common(5) if v > 1])
print("nume duplicat   :", [(k, v) for k, v in nume.most_common(5) if v > 1])

In [ ]:
scurte = sorted(rows, key=lambda r: len(r["description"]))[:5]
for r in scurte:
    print(len(r["description"]), "|", r["operational_name"], "|", r["description"])

In [ ]:
site = Counter(r["website"] for r in rows if r["website"])
nume = Counter(r["operational_name"] for r in rows if r["operational_name"])

print("website duplicat:", [(k, v) for k, v in site.most_common(10) if v > 1])
print("nume duplicat   :", [(k, v) for k, v in nume.most_common(10) if v > 1])

In [ ]:
from collections import defaultdict

grupuri = defaultdict(list)
for i, r in enumerate(rows):
    if r["operational_name"]:
        grupuri[r["operational_name"]].append(i)

dupe = {k: v for k, v in grupuri.items() if len(v) > 1}

print("nume cu duplicate:", len(dupe))
print("rânduri în plus  :", sum(len(v) - 1 for v in dupe.values()))

identice = 0
for nume_c, idx in dupe.items():
    prim = rows[idx[0]]
    if all(rows[j] == prim for j in idx[1:]):
        identice += 1
    else:
        print("\nDIFERĂ:", nume_c)
        for j in idx:
            r = rows[j]
            print(f"  [{j}] {r['website']} | {r['address'].get('country_code')} | "
                  f"{r['primary_naics']['code']} | emp={r['employee_count']} | "
                  f"desc={len(r['description'])} car.")

print(f"\ngrupuri identice: {identice}/{len(dupe)}")

In [ ]:
print(Counter(v for r in rows for v in r["business_model"]).most_common())
for r in rows:
    if "Logistics/Transportation" in r["business_model"]:
        print(r["operational_name"], "|", r["address"]["country_code"], "|", r["description"][:150])

In [ ]:
fr = [r for r in rows if r["address"]["country_code"] == "fr"]
print("companii franceze:", len(fr))

food_naics = [r for r in fr if r["primary_naics"]["code"][:3] in ("311", "312")]
print("cu NAICS 311/312:", len(food_naics))

kw = ["food", "beverage", "drink", "wine", "dairy", "bakery", "brewing", "agri"]
food_text = [r for r in fr
             if any(k in (r["description"] + " " + " ".join(r["core_offerings"])).lower()
                    for k in kw)]
print("cu cuvinte cheie:", len(food_text))

coduri_naics = {id(r) for r in food_naics}
print("\nîn text dar nu în NAICS:", len([r for r in food_text if id(r) not in coduri_naics]))

In [ ]:
amb = [r for r in rows if "packaging" in (r["description"] + " " + " ".join(r["core_offerings"])).lower()]
for r in amb:
    print(r["target_markets"], "|", r["operational_name"], "|", r["description"][:100])